# Trastuzumab response prediction

Trains a 5-fold cross-validated logistic-regression ensemble on TransNEO
trastuzumab, **plus** a per-fold MinMax-scaled ERBB2 spatial-heterogeneity score
(SPAND). Reports AUCs for three predictors — clusters only, ERBB2-only, and
their average — on TransNEO 5-fold CV and on the IMPRESS, PBCP and Cedars-Sinai external tests.
Adapted from `scr/new_tras_clusters/auc_final_trastuzumab_new.py`.

**Inputs**
- `../../clustering/data/transneo_trastuzumab_cluster_props.csv` — 61 TransNEO trastuzumab slides.
- `../../clustering/data/impress_trastuzumab_cluster_props.csv` — 62 IMPRESS trastuzumab slides.
- `../../clustering/data/pbcp_trastuzumab_cluster_props.csv` — 18 PBCP trastuzumab slides.
- `../../clustering/data/cedars_sinai_cluster_props.csv` — 30 Cedars-Sinai trastuzumab slides.
- `../data/response_labels.csv` — pCR / non-pCR per slide.
- `../data/erbb2_spand_scores.csv` — per-slide ERBB2 SPAND scores (`y_pred_proba` from the ERBB2-SPAND model, sign-flipped).

**Outputs**
- `../models/trastuzumab_cluster_ensemble.joblib` — cluster-based 5-fold ensemble.
- `../models/trastuzumab_combined.joblib` — `CombinedModel` averaging cluster proba with the scaled ERBB2 score.

## Setup

In [1]:
import sys
sys.path.append("../lib")

import joblib
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score

from model_classes import EnsembleModel, FeatureAligner, CombinedModel

import warnings; warnings.filterwarnings("ignore")

def per_patient_props(per_domain_csv):
    """Sum proportion_of_spots per (slide_name, predicted_cluster), then pivot."""
    d = pd.read_csv(per_domain_csv)
    return (d.groupby(['slide_name', 'predicted_cluster'])['proportion_of_spots']
              .sum().reset_index()
              .pivot(index='slide_name', columns='predicted_cluster', values='proportion_of_spots')
              .fillna(0))

/home/shulmaned/.local/lib/python3.10/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.7.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


## Load training data

In [2]:
labels = pd.read_csv("../data/response_labels.csv", index_col=0); labels.index = labels.index.astype(str)
erbb2  = pd.read_csv("../data/erbb2_spand_scores.csv", index_col=0)
print("ERBB2 cohorts:", erbb2["Cohort"].value_counts().to_dict())

X_train = per_patient_props("../../clustering/data/transneo_trastuzumab_per_domain.csv")
X_train.index = X_train.index.astype(str)
keep = X_train.index.intersection(labels[labels["Response"].notna()].index)
X_train = X_train.loc[keep]
y_train = labels.loc[keep, "Response"].astype(int).values
print(f"TransNEO trastuzumab: {X_train.shape[0]} slides, {X_train.shape[1]} features, prevalence={y_train.mean():.3f}")

het_train = erbb2[erbb2["Cohort"] == "TransNEO (Cross-validation)"]
X_het_train = X_train.join(het_train[["her2_spand"]], how="left")[["her2_spand"]]
print(f"ERBB2 SPAND on train: {X_het_train.shape}, NaN={X_het_train['her2_spand'].isna().sum()}")

ERBB2 cohorts: {'IMPRESS (External Validation set)': 62, 'TransNEO (Cross-validation)': 61}
TransNEO trastuzumab: 61 slides, 9 features, prevalence=0.311
ERBB2 SPAND on train: (61, 1), NaN=0


## Train — 5-fold stratified CV (cluster + ERBB2 scalers)

In [3]:
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=1)
param_grid = {
    "select__k":      ["all"],
    "logreg__penalty": ["l1"],
    "logreg__C":       [100],
    "logreg__solver":  ["saga"],
}

models, cv_scores, cv_y, cv_idx = [], [], [], []
scalers_het, cv_scores_het = [], []
for tr, te in kf.split(X_train, y_train):
    pipe = Pipeline([
        ("scaler", MinMaxScaler()),
        ("select", SelectKBest(f_classif)),
        ("logreg", LogisticRegression(max_iter=10000, class_weight="balanced")),
    ])
    gs = GridSearchCV(pipe, param_grid, cv=5, scoring="roc_auc",
                      refit="logreg__C").fit(X_train.iloc[tr], y_train[tr])
    models.append(gs.best_estimator_)
    proba = gs.best_estimator_.predict_proba(X_train.iloc[te])[:, 1]
    cv_scores.extend(proba); cv_y.extend(y_train[te]); cv_idx.extend(X_train.iloc[te].index)

    s = MinMaxScaler().fit(X_het_train.iloc[tr])
    scalers_het.append(s)
    cv_scores_het.extend(s.transform(X_het_train.iloc[te]).ravel())

cv_pred = pd.DataFrame({
    "y_true":           cv_y,
    "proba_cluster":    cv_scores,
    "proba_erbb2":      np.clip(np.array(cv_scores_het), 0, 1),
}, index=cv_idx)
cv_pred["proba_combined"] = (cv_pred["proba_cluster"] + cv_pred["proba_erbb2"]) / 2

print(f"TransNEO 5-fold CV cluster  AUC = {roc_auc_score(cv_pred['y_true'], cv_pred['proba_cluster']):.4f}")
print(f"TransNEO 5-fold CV ERBB2    AUC = {roc_auc_score(cv_pred['y_true'], cv_pred['proba_erbb2']):.4f}")
print(f"TransNEO 5-fold CV combined AUC = {roc_auc_score(cv_pred['y_true'], cv_pred['proba_combined']):.4f}")

TransNEO 5-fold CV cluster  AUC = 0.8672
TransNEO 5-fold CV ERBB2    AUC = 0.7782
TransNEO 5-fold CV combined AUC = 0.8997


## Save the cluster and combined models

In [4]:
ensemble_pipeline = Pipeline([
    ("aligner", FeatureAligner(expected_features=list(X_train.columns),
                                fill_values=X_train.mean().to_dict())),
    ("ensemble", EnsembleModel(models=models)),
])
joblib.dump(ensemble_pipeline, "../models/trastuzumab_cluster_ensemble.joblib")

combined_model = CombinedModel(ensemble_pipeline=ensemble_pipeline,
                                scalers_het=scalers_het,
                                spand_feature="her2_spand")
joblib.dump(combined_model, "../models/trastuzumab_combined.joblib")
print("Saved trastuzumab_cluster_ensemble.joblib and trastuzumab_combined.joblib")

Saved trastuzumab_cluster_ensemble.joblib and trastuzumab_combined.joblib


## Evaluate on the external cohorts

For each test cohort we compute three AUCs: cluster only, ERBB2 only (per the
paper's `1 - mean(scaled)` formula for test cohorts), and their average.

In [5]:
het_test = erbb2[erbb2["Cohort"] == "IMPRESS (External Validation set)"]
het_test_index = het_test.index

def evaluate(per_domain_csv, cohort_name, erbb2_flip=False):
    Xc = per_patient_props(per_domain_csv); Xc.index = Xc.index.astype(str)
    common = Xc.index.intersection(labels[labels["Response"].notna()].index)
    if len(common) == 0:
        return pd.DataFrame()
    Xc = Xc.loc[common]
    yc = labels.loc[common, "Response"].astype(int).values

    proba_clust = ensemble_pipeline.predict_proba(Xc)

    if cohort_name == "IMPRESS":
        # ERBB2 only — use scalers, apply 1-x flip per script
        het_idx = Xc.index.intersection(het_test_index)
        X_h = het_test.loc[het_idx, ["her2_spand"]]
        scaled = np.array([s.transform(X_h) for s in scalers_het]).mean(axis=0).mean(axis=1)
        scaled = np.clip(1 - scaled, 0, 1)
        proba_erbb2 = pd.Series(scaled, index=het_idx).reindex(Xc.index).values
        proba_combined = np.nanmean(np.vstack([proba_clust, proba_erbb2]), axis=0)
    else:
        proba_erbb2 = np.full(len(Xc), np.nan)
        proba_combined = proba_clust

    def safe_auc(y, p):
        m = ~np.isnan(p)
        return roc_auc_score(y[m], p[m]) if m.sum() and len(np.unique(y[m])) > 1 else float("nan")

    auc_c = safe_auc(yc, proba_clust)
    auc_e = safe_auc(yc, proba_erbb2)
    auc_x = safe_auc(yc, proba_combined)
    print(f"{cohort_name:<10} n={len(yc):>3}, prev={yc.mean():.3f}  cluster={auc_c:.4f}  erbb2={auc_e:.4f}  combined={auc_x:.4f}")
    return pd.DataFrame({"y_true": yc, "proba_cluster": proba_clust,
                          "proba_erbb2": proba_erbb2, "proba_combined": proba_combined,
                          "Cohort": cohort_name}, index=common)

print(f"TransNEO   n={len(cv_pred):>3}, prev={cv_pred['y_true'].mean():.3f}  cluster={roc_auc_score(cv_pred['y_true'],cv_pred['proba_cluster']):.4f}  erbb2={roc_auc_score(cv_pred['y_true'],cv_pred['proba_erbb2']):.4f}  combined={roc_auc_score(cv_pred['y_true'],cv_pred['proba_combined']):.4f}")
df_impress = evaluate("../../clustering/data/impress_trastuzumab_per_domain.csv", "IMPRESS")
df_pbcp    = evaluate("../../clustering/data/pbcp_trastuzumab_per_domain.csv",    "PBCP")
df_cedars  = evaluate("../../clustering/data/cedars_sinai_per_domain.csv",       "Cedars-Sinai")

cv_pred["Cohort"] = "TransNEO (5-fold CV)"
all_pred = pd.concat([cv_pred, df_impress, df_pbcp, df_cedars])
all_pred.head()

TransNEO   n= 61, prev=0.311  cluster=0.8672  erbb2=0.7782  combined=0.8997
IMPRESS    n= 62, prev=0.613  cluster=0.7434  erbb2=0.2741  combined=0.3684
PBCP       n= 18, prev=0.611  cluster=0.8961  erbb2=nan  combined=0.8961
Cedars-Sinai n= 30, prev=0.867  cluster=0.6827  erbb2=nan  combined=0.6827


,y_true,proba_cluster,proba_erbb2,proba_combined,Cohort
BC_00000_470650,0,0.285078,0.897645,0.591362,TransNEO (5-fold CV)
BC_00006_470656,0,0.099944,0.924816,0.512380,TransNEO (5-fold CV)
BC_00021_471096,1,0.943810,0.917181,0.930495,TransNEO (5-fold CV)
BC_00056_473356,1,0.906857,0.926181,0.916519,TransNEO (5-fold CV)
BC_00087_631901,1,0.760248,0.919903,0.840076,TransNEO (5-fold CV)


## AUC summary

In [6]:
def safe_auc(y, p):
    p = np.array(p)
    m = ~np.isnan(p)
    return roc_auc_score(np.array(y)[m], p[m]) if m.sum() and len(np.unique(np.array(y)[m])) > 1 else float("nan")

rows = []
for c, g in all_pred.groupby("Cohort"):
    rows.append({
        "Cohort": c, "n": len(g),
        "Prevalence":   g["y_true"].mean(),
        "AUC cluster":  safe_auc(g["y_true"], g["proba_cluster"]),
        "AUC ERBB2":    safe_auc(g["y_true"], g["proba_erbb2"]),
        "AUC combined": safe_auc(g["y_true"], g["proba_combined"]),
    })
summary = pd.DataFrame(rows).sort_values("Cohort").reset_index(drop=True)
summary.round(3)

,Cohort,n,Prevalence,AUC cluster,AUC ERBB2,AUC combined
0,Cedars-Sinai,30,0.867,0.683,NaN,0.683
1,IMPRESS,62,0.613,0.743,0.274,0.368
2,PBCP,18,0.611,0.896,NaN,0.896
3,TransNEO (5-fold CV),61,0.311,0.867,0.778,0.900
